In [1]:
from mxnet import nd, gluon, init, autograd, gpu
from mxnet.gluon import nn
import mxnet as mx

import numpy as np
import pandas as pd

import os
from os import listdir
from os import makedirs
import time
from random import randint
import datetime

# 0.1.1: Constants

In [2]:
root_data = 'D:/Work/2022/FCNN/02.15/Data/1/20/1.000000/'
root_results = 'C:/Users/ossid/GoogleDrive/Work/Research/2020/Stepan/10.21/'

rawdata_path = root_data + 'Symbols_1m_1ch_PR_'
results_path = root_results + 'Logs/Complex/'

gray_symbols_16qam = np.sqrt(0.1) * np.array(
                    [1+1j, 1+3j, 1-1j, 1-3j, 
                    3+1j, 3+3j, 3-1j, 3-3j, 
                    -1+1j, -1+3j, -1-1j, -1-3j,
                    -3+1j, -3+3j, -3-1j, -3-3j])

swap_64_to_128_complex = False

if swap_64_to_128_complex:
    complex_t = np.complex128
    real_t = np.float64
else:
    complex_t = np.complex64
    real_t = np.float32

# 0.1.2: Variables

In [3]:
# 1: Data import
data_size = 524288
data_files = 32
train_portion = 0.97
train_files = int(data_files * train_portion)
test_files = data_files - train_files

# 2.1: Create model
radius = 10
sample_size = data_size - 2 * radius

shape_size = 4 * radius + 2
neuron_hidden_layer = 32

init_seed = randint(0, 100000000)

# 3: Fit model
epochs = 50
minibatch_multiplier = 1
batch_size = data_size * minibatch_multiplier

lrate = 0.001

save_model_best = True
model_filename = "net.params"

continue_train = False
model_path = 'C:/Users/ossid/GoogleDrive/Work/Research/2020/Stepan/10.21/Logs/Complex/20210129_1032_15433378/'

calc_ber = True

run_options = "10x32 Shodimost LR reduce 20 decay steps Start 0.001"

print('Seed:', init_seed)
print('Batch size:', batch_size)

mx.random.seed(init_seed)

Seed: 53227027
Batch size: 524288


# 0.2: Functions

# 1: Data import
> data_size, data_files, train_portion; 
> train_files, test_files

In [4]:
def symbols_to_codes_v2(data_complex):
    codes = np.zeros(len(data_complex), dtype=np.int32)
    for i in range(len(data_complex)):
        codes[i] = np.argmin(abs(gray_symbols_16qam - data_complex[i]))
    
    return codes

def ber_by_codes(tx, rx):
    diff = tx ^ rx
    errors = 0
    for error in diff:
        while error:
            error &= error - 1
            errors +=1
            
    return 0.25 * errors / len(diff)

def create_metadata():
    metadata = dict( (name, eval(name)) for name in ['init_seed', 'data_size','data_files','train_portion', 'train_files', 'test_files',
                                     'radius', 'neuron_hidden_layer', 'lrate', 'epochs', 'minibatch_multiplier', 'batch_size', 'continue_train', 'swap_64_to_128_complex'] )
    now = datetime.datetime.now()
    date_and_time = now.strftime('%Y%m%d_%H%M')
    file_signature = date_and_time + '_' + str(init_seed)
    dir_path = results_path + file_signature + '/'
    makedirs(dir_path)
    
    f = open(dir_path + 'metadata.md', 'w')
    f.write(now.strftime('%Y-%m-%d %H:%M') + '\n\n')
    f.write('Complex NN\n\n')
    f.write('{}\n\n'.format(run_options))
    
    for x, y in metadata.items():
        f.write('{}: {}\n'.format(x, y))
    f.close()
    
    return dir_path

# Gluon dataloader method

In [5]:
def dataloader_CNN(X, y, batch, radius, enableShuffle):
    size = X.shape[1]
    batch = min(batch, size)
    d = size / batch
    if d%1 > 0.2:
        batch = np.floor(size / np.ceil(d)).astype(int)
    batch = batch - batch%2
    number_of_bathes = np.floor(size / batch).astype(int)
    
    data = nd.empty((number_of_bathes, 2, batch))
    label = nd.empty((number_of_bathes, 2, batch - 2*radius))
    for i in range(number_of_bathes):
        data[i,:,:] = nd.array(X[:,i*batch:(i+1)*batch])
        for j in range(2):
            label[i,j:(j+1),:] = nd.array(y[j:(j+1),i*batch + radius:(i+1)*batch - radius])
    dataset = gluon.data.dataset.ArrayDataset(data, label)
    return gluon.data.DataLoader(dataset, batch_size=1, shuffle=enableShuffle), batch, number_of_bathes

In [6]:
time_start = time.time()

X_train = np.zeros((train_files * data_size, 2), dtype=real_t)
y_train = np.zeros((train_files * data_size, 2), dtype=real_t)
y_train_ber = np.zeros((train_files * sample_size, 2), dtype=real_t)
X_test = np.zeros((test_files * data_size, 2), dtype=real_t)
y_test = np.zeros((test_files * data_size, 2), dtype=real_t)
y_test_ber = np.zeros((test_files * sample_size, 2), dtype=real_t)

columns = ['tx_re', 'tx_im', 'rx_re', 'rx_im']

for i in range(train_files):
    data = pd.read_csv(rawdata_path + str(i + 1) + '.csv', names=columns).to_numpy(dtype=real_t) 
    X_train[i * data_size:(i + 1) * data_size,:] = data[:,2:4]
    y_train[i * data_size:(i + 1) * data_size] = data[:,0:2]
    y_train_ber[i * sample_size:(i + 1) * sample_size] = data[radius:data_size-radius,0:2]
    
for i in range(test_files):
    data = pd.read_csv(rawdata_path + str(train_files + i + 1) + '.csv', names=columns).to_numpy(dtype=real_t) 
    X_test[i * data_size:(i + 1) * data_size,:] =data[:,2:4]
    y_test[i * data_size:(i + 1) * data_size] = data[:,0:2]
    y_test_ber[i * sample_size:(i + 1) * sample_size] = data[radius:data_size-radius,0:2]

del data

X_train_loader = np.transpose(X_train)
y_train_loader = np.transpose(y_train)
X_test_loader = np.transpose(X_test)
y_test_loader = np.transpose(y_test)

[train_dataloader_CNN, batch_train, number_of_bathes_train] = dataloader_CNN(X_train_loader, y_train_loader, batch_size, radius, True)
[test_dataloader_CNN, batch_test, number_of_bathes_test]  = dataloader_CNN(X_test_loader, y_test_loader, batch_size, radius, False)

del X_train_loader, y_train_loader, X_test_loader, y_test_loader

time_end = time.time()
print('Time elapsed: {}'.format(time_end - time_start))

Time elapsed: 7.384113311767578


# 2.1: Create and compile model
> radius, sample_size; 
> shape_size, neuron_hidden_layer; 
> init_seed

In [7]:
class ComplexDenseCNN_v1(gluon.Block):
    def __init__(self, channels, units, inputs, kernel_size, strides=1, **kwargs):
        super(ComplexDenseCNN_v1, self).__init__()
        self.units = units
        self.inputs = inputs
        self.channels = channels
        with self.name_scope():
            self.conv_dense_re = gluon.nn.Conv1D(channels=units, in_channels=inputs, groups=1, kernel_size=kernel_size, strides=strides, activation=None, use_bias=False)
            self.conv_dense_im = gluon.nn.Conv1D(channels=units, in_channels=inputs, groups=1, kernel_size=kernel_size, strides=strides, activation=None, use_bias=False)
    def forward(self, z):
       
        u = self.conv_dense_re(z[:,0:self.inputs,:]) - self.conv_dense_im(z[:,self.inputs:2*self.inputs,:])
        v = self.conv_dense_im(z[:,0:self.inputs,:]) + self.conv_dense_re(z[:,self.inputs:2*self.inputs,:])
        
        return nd.concat(u, v, dim=1)

In [8]:
class KerrActivationEnhanced_1ch_v1(gluon.Block):
    def __init__(self, units, **kwargs):
        super(KerrActivationEnhanced_1ch_v1, self).__init__()
        self.units = units
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(1,))

    def forward(self, z):
        power = self.conv_intra.data() * (nd.square(z[:,0:self.units,:]) +  nd.square(z[:,self.units:,:]))      
        cos = nd.cos(power)
        sin = nd.sin(power)

        u = cos * z[:,0:self.units,:] - sin * z[:,self.units:,:]
        v = cos * z[:,self.units:,:] + sin * z[:,0:self.units,:]
       
        return nd.concat(u, v, dim=1)
    
    
    
class KerrActivationEnhanced_1ch_v2(gluon.Block):
    def __init__(self, units, **kwargs):
        super(KerrActivationEnhanced_1ch_v2, self).__init__()
        self.units = units
        with self.name_scope():
            self.conv_intra = gluon.nn.Conv1D(channels=units, in_channels=units, groups=units, kernel_size=1, strides=1, activation=None, use_bias=False)

    def forward(self, z):
        power = self.conv_intra(nd.square(z[:,0:self.units,:]) +  nd.square(z[:,self.units:,:]))
        
        cos = nd.cos(power)
        sin = nd.sin(power)

        u = cos * z[:,0:self.units,:] - sin * z[:,self.units:,:]
        v = cos * z[:,self.units:,:] + sin * z[:,0:self.units,:]
        
        return nd.concat(u, v, dim=1)

In [9]:
ctx = gpu(0)

net = gluon.nn.Sequential()

with net.name_scope():
    net.add(ComplexDenseCNN_v1(channels=2,units=neuron_hidden_layer,inputs=1,kernel_size=2*radius+1))
    net.add(KerrActivationEnhanced_1ch_v1(units=neuron_hidden_layer))
    net.add(ComplexDenseCNN_v1(channels=2,units=neuron_hidden_layer,inputs=neuron_hidden_layer,kernel_size=1))
    net.add(KerrActivationEnhanced_1ch_v1(units=neuron_hidden_layer))
    net.add(ComplexDenseCNN_v1(channels=2,units=1,inputs=neuron_hidden_layer,kernel_size=1))
    net.add(KerrActivationEnhanced_1ch_v1(units=1))


mse = gluon.loss.L2Loss()

model_saved = False
min_loss = 100
decay_steps = 20
steps_wo_min = 0

if continue_train:
    net.load_parameters(model_path + model_filename, ctx=ctx)
    
    f = open(model_path + 'learning_rate.dat', 'r')
    lrate = float(f.read())
    f.close()
    f = open(model_path + 'reached_loss.dat', 'r')
    min_loss = float(f.read())
    f.close()
    f = open(model_path + 'steps_wo_min.dat', 'r')
    steps_wo_min = float(f.read())
    f.close()
else:
    net.initialize(init=mx.init.Normal(sigma=0.05), ctx=ctx)
    
trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate})
if continue_train:
    trainer.load_states(model_path + 'optimizer.state')

# 3: Fit the model
> epochs, minibatch_multiplier, batch_size, file_step

In [10]:
dir_path = create_metadata()

filename = os.path.join(dir_path, model_filename)
filename_optimizer = os.path.join(dir_path, 'optimizer.state')

f = open(dir_path + 'learning_rate.dat', 'w')
f.write('{}\n'.format(lrate))
f.close()

f = open(dir_path + 'reached_loss.dat', 'w')
f.write('{}\n'.format(min_loss))
f.close()

f = open(dir_path + 'nn_train.dat', 'a+')

glob_start = time.time()

for epoch in range(epochs):
    train_loss = nd.zeros(1, ctx=ctx)
    tic = time.time()
    for data, label in train_dataloader_CNN:
        data = data.as_in_context(ctx)
        label = label.as_in_context(ctx)
        with autograd.record():
            output = net(data)
            loss = mse(output, label)
            loss = nd.mean(loss)
        
        loss.backward()
        trainer.step(1)
        train_loss += loss.mean().asscalar()
    
    if (epoch) % 5 == 0:
        print('epoch', epoch + 1, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
        f.write('epoch {} -- loss {} -- time {}\n'.format(epoch + 1, train_loss.asscalar(), time.time()-tic))
        
    if min_loss > train_loss.asscalar():
        steps_wo_min = 0
        min_loss = train_loss.asscalar()
        if save_model_best:
            net.save_parameters(filename)
            trainer.save_states(filename_optimizer)
            flr = open(dir_path + 'reached_loss.dat', 'w')
            flr.write('{}\n'.format(min_loss))
            flr.close()
            model_saved = True
            print('epoch', epoch + 1, '-- Model saved', '-- loss', train_loss.asscalar())
            f.write('epoch {} -- Model saved -- loss {}\n'.format(epoch + 1, train_loss.asscalar()))
    else:
        steps_wo_min += 1
        
    if steps_wo_min >= decay_steps:
        steps_wo_min = 0
        lrate /= 2
        flr = open(dir_path + 'learning_rate.dat', 'w')
        flr.write('{}\n'.format(lrate))
        flr.close()
        if lrate < 1e-4:
            print('Train finished!')
            break
        trainer.set_learning_rate(lrate)
        print('epoch', epoch + 1, '-- Learning rate', lrate)
        f.write('epoch {} -- Learning rate {}\n'.format(epoch + 1, lrate))
        
f.close()
output.wait_to_read() 

if model_saved == False:
    shutil.rmtree(dir_path)
    f = open(model_path + 'steps_wo_min.dat', 'w')
    f.write('{}\n'.format(steps_wo_min))
    f.close()
    flr = open(model_path + 'learning_rate.dat', 'w')
    flr.write('{}\n'.format(lrate))
    flr.close()
else:
    f = open(dir_path + 'steps_wo_min.dat', 'w')
    f.write('{}\n'.format(steps_wo_min))
    f.close()

glob_end = time.time()

print(time.time() - glob_start)

epoch 1 -- loss 5.835696 -- time 5.571394443511963
epoch 1 -- Model saved -- loss 5.835696
epoch 2 -- Model saved -- loss 0.9278956
epoch 3 -- Model saved -- loss 0.30808517
epoch 4 -- Model saved -- loss 0.28857768
epoch 5 -- Model saved -- loss 0.2877478
epoch 6 -- loss 0.28744033 -- time 2.576716184616089
epoch 6 -- Model saved -- loss 0.28744033
epoch 7 -- Model saved -- loss 0.28724533
epoch 8 -- Model saved -- loss 0.28718314
epoch 9 -- Model saved -- loss 0.28699154
epoch 10 -- Model saved -- loss 0.28683174
epoch 11 -- loss 0.28684366 -- time 4.000306844711304
epoch 12 -- Model saved -- loss 0.2865345
epoch 13 -- Model saved -- loss 0.28623992
epoch 14 -- Model saved -- loss 0.28592753
epoch 15 -- Model saved -- loss 0.28553587
epoch 16 -- loss 0.28518468 -- time 2.529242992401123
epoch 16 -- Model saved -- loss 0.28518468
epoch 17 -- Model saved -- loss 0.28440744
epoch 18 -- Model saved -- loss 0.2834346
epoch 19 -- Model saved -- loss 0.28216717
epoch 20 -- Model saved -- lo

# 4: Calculating BER

In [11]:
glob_end = time.time()

batch_size_n = batch_test - 2*radius

rx_test = symbols_to_codes_v2(X_test[:,0] + 1j * X_test[:,1])
tx_test_pred = nd.empty([1, 2, test_files * sample_size], ctx=ctx)

for batch_idx, data in enumerate(test_dataloader_CNN):
    data[0] = data[0].as_in_context(ctx)
    output = net(data[0])
    if (output.shape[2] < batch_size_n):
        tx_test_pred[:,:,batch_idx*batch_size_n:] = output
    else:
        tx_test_pred[:,:,batch_idx*batch_size_n:(batch_idx+1)*batch_size_n] = output

tx_test_pred = tx_test_pred.as_in_context(mx.cpu(0)).asnumpy()
tx_test = symbols_to_codes_v2(y_test[:,0] + 1j * y_test[:,1])
tx_test_nn = symbols_to_codes_v2(y_test_ber[:,0] + 1j * y_test_ber[:,1])
ber_before_nn = ber_by_codes(tx_test, rx_test)
ber_after_nn = ber_by_codes(tx_test_nn, symbols_to_codes_v2(tx_test_pred[0,0,:] + 1j * tx_test_pred[0,1,:]))

# Train BER

batch_size_n = batch_train - 2*radius

rx_train = symbols_to_codes_v2(X_train[:,0] + 1j * X_train[:,1])
tx_train_pred = nd.empty([1, 2, train_files * sample_size], ctx=ctx)
y_train_shuffled = nd.empty([1, 2, train_files * sample_size], ctx=ctx)
for batch_idx, data in enumerate(train_dataloader_CNN):
    data[0] = data[0].as_in_context(ctx)
    output = net(data[0])
    y_train_shuffled[:,:,batch_idx*batch_size_n:(batch_idx+1)*batch_size_n] = data[1]
    if (output.shape[2] < batch_size_n):
        tx_train_pred[:,:,batch_idx*batch_size_n:] = output
    else:
        tx_train_pred[:,:,batch_idx*batch_size_n:(batch_idx+1)*batch_size_n] = output

tx_train_pred = tx_train_pred.as_in_context(mx.cpu(0)).asnumpy()
y_train_shuffled = y_train_shuffled.as_in_context(mx.cpu(0)).asnumpy()
tx_train = symbols_to_codes_v2(y_train[:,0] + 1j * y_train[:,1])
tx_train_nn = symbols_to_codes_v2(y_train_ber[:,0] + 1j * y_train_ber[:,1])
tx_train_nn_shuffled = symbols_to_codes_v2(y_train_shuffled[0,0,:] + 1j * y_train_shuffled[0,1,:])
ber_before_nn_train = ber_by_codes(tx_train, rx_train)
ber_after_nn_train = ber_by_codes(tx_train_nn_shuffled, symbols_to_codes_v2(tx_train_pred[0,0,:] + 1j * tx_train_pred[0,1,:]))

###############################################################################

f = open(dir_path + 'metadata.md', 'a')
f.write('\nTime elapsed: {}\n'.format(glob_end - glob_start))

f.write('Epochs to decay: {}\n'.format(decay_steps))
if epochs > 0:
    f.write('\nEpochs total: {}\n'.format(epoch + 1))
else:
    f.write('\nEpochs total: {}\n'.format(0))
f.write('Minimum loss: {}\n'.format(min_loss))

f.write('\nTest BER:\n==================\n\n')

f.write('\nBER before NN: {}\n'.format(ber_before_nn))
f.write('BER after NN: {}\n'.format(ber_after_nn))

f.write('\nTrain BER:\n==================\n\n')
f.write('BER before NN: {}\n'.format(ber_before_nn_train))
f.write('BER after NN: {}\n'.format(ber_after_nn_train))

f.close()

f = open(dir_path + 'metadata.md', 'r')
for line in f:
    print(line, end = '')
f.close()

print(time.time() - glob_start)

2022-02-18 14:16

Complex NN

10x32 Shodimost LR reduce 20 decay steps Start 0.001

init_seed: 53227027
data_size: 524288
data_files: 32
train_portion: 0.97
train_files: 31
test_files: 1
radius: 10
neuron_hidden_layer: 32
lrate: 0.001
epochs: 50
minibatch_multiplier: 1
batch_size: 524288
continue_train: False
swap_64_to_128_complex: False

Time elapsed: 140.11567759513855
Epochs to decay: 20

Epochs total: 50
Minimum loss: 0.22751201689243317

Test BER:


BER before NN: 0.009646892547607422
BER after NN: 0.004768076632561972

Train BER:

BER before NN: 0.009839627050584363
BER after NN: 0.004814362366255919
438.4880232810974


In [12]:
# params = net.collect_params()
# print(params)